# RBDisco API Example
RBDisco is a plugin to ActiTect to predict RBD status from multi-night actigraphy recordings. It hosts the pretrained models as described in the main [README](../../README.md)/paper. We provide batch processing via the CLI, but you can also use `actitect.rbdisco.predict()` for more flexibility.

In [ ]:

from actitect.api import ensure_demo_asset
example_file_path = ensure_demo_asset()  # choose any filepath pointing to a supported file. Here, we download a provided example.

from actitect.rbdisco import predict
from actitect.api import process, compute_sleep_motor_features

SUBJECT_ID = 'ID-1'

processed_df, _ = process(example_file_path, subject_id=SUBJECT_ID) # use actitect to load and preprocess the data
feat_df = compute_sleep_motor_features(processed_df, subject_id=SUBJECT_ID)

night_level = predict(
    data=feat_df,  # we use the extracted sleep-motor features for prediction; you can also directly pass a pathlike object pointing to a valid actigraphy binary (.cwa, .bin, .gt3x or .csv) and it will run the features extraction under the hood
	return_level='night',  # this will return nightly scores and predictions
    subject_id=SUBJECT_ID  # optional
)

print(night_level)

/Users/david/anaconda3/envs/RBDisco/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 (17:05:23) - [actitect.api|INFO]: File' example.cwa' already exists, skipping download.
 (17:05:23) - [actitect.actimeter.factory|INFO]: (io: ID-1) detected 'AxivityAx6' device.
 (17:05:23) - [actitect.actimeter.basedevice|INFO]: (io: ID-1) parsing data from 'example.cwa'.
 (17:05:33) - [actitect.utils.data_utils|INFO]: (io: ID-1) found 38 duplicate timestamps.removing ~0.38s of data
 (17:05:35) - [actitect.actimeter.basedevice|INFO]: (io: ID-1) successfully loaded raw data. (11.61s)
 (17:05:35) - [actitect.actimeter.basedevice|INFO]: (resampling: ID-1) non-uniform sampling rate in raw data detected: fs = 100.05 ± 24.85 Hz


Here, each row corresponds to one night of the recording, assigned to a specific RBD probability score in [0,1] and a thresholded binary prediction. To account for night-to-night variability and increase prediction robustness, we aggregate the per-night predictions to patient level (details in paper):

In [ ]:
patient_level = predict(
    data=feat_df,  # again, you could also simply pass 'file' here to automatically compute the features, but here we want to use the cached features
	return_level='patient',  # this is the default: nightly scores are aggregated to patient level
    subject_id=SUBJECT_ID  # optional
)

print(patient_level)

By default, `.predict()` will use the pretrained multi-center as an estimator. You can also use the single-center model (refer to paper) by specifying:
```
patient_level = predict(
    data=file,
	return_level='patient',
	model='singleCenterCore',  # additional arg to use the single-center model instead
    subject_id='ID-1'  # optional
)
```